# Bonus — Natural-Language Q&A for the Museum Guide
Zero-shot cosine similarity with MiniLM-L6-v2 (HuggingFace) and a confidence guardrail.

1. **Step 0**: build the artwork index (run once offline — embed all 1 500 captions).
2. **Step 1**: demo queries — the subject's own example questions.
3. **Step 2**: guardrail in action — a query that triggers "not sure".
4. **Step 3**: τ sweep — coverage vs precision trade-off.
5. **Conclusion**: answer the director's question.

## How to run

### Option A — Local (recommended for development)
**Kernel**: select `.venv-qa` in VS Code (bottom-right kernel picker), not the project's `.venv`.

`.venv-qa` is a separate virtual environment — required because the project's main `.venv` pins
`torch==2.2.2` which conflicts with `transformers`. Create it once from the repo root:

```bash
python3 -m venv .venv-qa
source .venv-qa/bin/activate
pip install "torch>=2.6" transformers pandas
```

### Option B — Colab with GPU (recommended for building the index faster)
GPU (*Runtime → Change runtime type → T4 GPU*) cuts index build time from ~3 min to ~20 sec.
Uncomment and run the cell below first. Adapt `DATA_ON_DRIVE` to your Drive path.

In [ ]:
#  --- Colab only (Option B) — skip if running locally ---
# !pip install -q transformers
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_ON_DRIVE  = "/content/drive/MyDrive/wikiart_style5"          # <- adapt: unzipped folder or zip
# CAPS_ON_DRIVE  = "/content/drive/MyDrive/artwork_captions.json"   # generated in Step 0
# INDEX_ON_DRIVE = "/content/drive/MyDrive/artwork_index.pt"        # generated in Step 0, save back to Drive after
# !git clone -q https://github.com/raniamehria/museum-guide.git /content/museum-guide
# %cd /content/museum-guide
# # If DATA_ON_DRIVE is a zip file:
# # !unzip -q "{DATA_ON_DRIVE}" -d data/
# # If DATA_ON_DRIVE is already an unzipped folder:
# !rm -rf data/wikiart_style5 && mkdir -p data && cp -r "{DATA_ON_DRIVE}" data/wikiart_style5
# !cp "{CAPS_ON_DRIVE}" data/artwork_captions.json
# !mkdir -p checkpoints && cp "{INDEX_ON_DRIVE}" checkpoints/artwork_index.pt  # skip if building fresh
# !find data/wikiart_style5 -name "*.jpg" | wc -l   # must print 1500

## Setup
Imports and working directory. Run from the repo root (the `chdir` below handles it automatically).

In [ ]:
import os
import sys
import json
from pathlib import Path

if os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")   # run from repo root so data/ and checkpoints/ resolve correctly
sys.path.insert(0, "src")

import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoModel, AutoTokenizer

from museum_guide.qa import build_artwork_index, answer, MODEL_NAME, INDEX_PATH

IMG_DIR = Path("data/wikiart_style5")

print("working dir:", os.getcwd())
print("torch:", torch.__version__)

## Step 0 — Build the artwork index (run once, offline)
Embeds all 1 500 artwork captions with MiniLM and saves `checkpoints/artwork_index.pt`.
Skip this cell if `artwork_index.pt` already exists (it takes ~3 min on CPU).

In [ ]:
if not Path(INDEX_PATH).exists():
    print("Building index...")
    build_artwork_index()
else:
    print(f"Index already exists at {INDEX_PATH} — skipping build.")

## Load the pre-built index and model
Load the tokenizer and encoder once — reused for all queries below.

In [ ]:
index = torch.load(INDEX_PATH, weights_only=False)
print(f"Loaded index: {len(index)} artworks, embedding dim {index[0]['embedding'].shape[0]}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME).eval()
print("Model ready:", MODEL_NAME)

## Step 1 — Demo queries
The five queries include the subject's own example questions — the ones we cannot change.

In [ ]:
TAU = 0.30

DEMO_QUERIES = [
    "Which painting shows a stormy sea?",           # subject example
    "I would like to see portraits of children",    # subject example
    "Show me Impressionist art",
    "I'm looking for something calm and Japanese",
    "Show me still life paintings",
]


def show_results(query, result):
    """Print scores and display the top-3 images side by side."""
    print(f"\nQ: {query}")
    if not result["answered"]:
        print(f"   GUARDRAIL ({result['top_score']:.3f} < τ={TAU}): {result['message']}")
        return

    hits = result["results"]
    print(f"   top score: {result['top_score']}")
    fig, axes = plt.subplots(1, len(hits), figsize=(5 * len(hits), 4), squeeze=False)
    for ax, r in zip(axes[0], hits):
        img = Image.open(IMG_DIR / r["path"])
        ax.imshow(img)
        ax.set_title(
            f"#{r['rank']}  score={r['score']:.3f}\n{r['style']}\n{r['artist']}",
            fontsize=9,
        )
        ax.axis("off")
    fig.suptitle(query, fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()


for q in DEMO_QUERIES:
    result = answer(q, index, tokenizer, model, tau=TAU, top_k=3)
    show_results(q, result)

## Step 2 — Guardrail in action
A query that is completely outside our collection triggers the "not sure" response.

In [ ]:
OUT_OF_SCOPE_QUERIES = [
    "Show me contemporary photography",
    "I want to see sculptures",
    "Do you have any Renaissance paintings?",
]

for q in OUT_OF_SCOPE_QUERIES:
    result = answer(q, index, tokenizer, model, tau=TAU, top_k=3)
    status = "GUARDRAIL" if not result["answered"] else "answered"
    print(f"[{status}]  score={result['top_score']:.3f}  Q: {q}")
    if not result["answered"]:
        print(f"           -> {result['message']}")

## Step 3 — τ sweep: coverage vs precision
For each threshold τ, measure:
- **coverage**: fraction of held-out queries the guide answers (vs says "not sure")
- **Recall@3**: fraction where the correct style appears in the top-3 results

The held-out queries are the 8 evaluation queries from `data/qa_training.json` that have a style or artist label we can check against.

In [ ]:
import pandas as pd

# 8 held-out queries with ground-truth style labels
EVAL_QUERIES = [
    {"query": "Show me Baroque paintings",              "expected_style": "Baroque"},
    {"query": "I want to see Cubist art",               "expected_style": "Cubism"},
    {"query": "Show me Impressionist paintings",        "expected_style": "Impressionism"},
    {"query": "I like Pop Art",                         "expected_style": "Pop_Art"},
    {"query": "Show me Japanese woodblock prints",      "expected_style": "Ukiyo_e"},
    {"query": "Still life paintings please",            "expected_style": None},   # no single style
    {"query": "I would like to see work by Hokusai",    "expected_style": "Ukiyo_e"},
    {"query": "Show me paintings by Rubens",            "expected_style": "Baroque"},
]

rows = []
for tau in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
    covered, recall3 = 0, 0
    checkable = sum(1 for q in EVAL_QUERIES if q["expected_style"] is not None)
    for q in EVAL_QUERIES:
        res = answer(q["query"], index, tokenizer, model, tau=tau, top_k=3)
        if res["answered"]:
            covered += 1
            if q["expected_style"] and any(r["style"] == q["expected_style"] for r in res["results"]):
                recall3 += 1
    rows.append({
        "tau": tau,
        "coverage": f"{covered / len(EVAL_QUERIES):.0%}",
        "Recall@3 (style)": f"{recall3 / checkable:.0%}" if checkable else "N/A",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nChosen tau = 0.30  (balances coverage vs precision)")

## Save evaluation results

In [ ]:
import json
Path("results/metrics").mkdir(parents=True, exist_ok=True)
qa_eval = {
    "tau_chosen": 0.30,
    "tau_sweep": rows,
    "demo_queries": DEMO_QUERIES,
    "eval_queries_n": len(EVAL_QUERIES),
    "note": "Zero-shot cosine similarity with MiniLM-L6-v2; index built from all 1500 artwork captions."
}
Path("results/metrics/qa_eval.json").write_text(json.dumps(qa_eval, indent=2))
print("Saved results/metrics/qa_eval.json")

## Conclusion — answering the director's question

| Task | Question | Answer |
|---|---|---|
| Task 3 — Image classification | Is ResNet-18 worth it vs MLP? | **Yes** — +38 points, 25 ms on CPU |
| Bonus — Natural-language Q&A | Is fine-tuning worth it vs zero-shot? | **No** — zero-shot retrieval is sufficient |

**Why zero-shot works**: MiniLM-L6-v2 is pre-trained on 1 billion sentence pairs and already understands
natural language similarity. Our LLM-generated captions cover the vocabulary visitors use
(portrait, landscape, stormy, calm, Japanese) without any fine-tuning.

**Guardrail**: if max cosine score < τ = 0.30, the guide says *"I'm not sure"* instead of
returning a wrong answer. This is the same NVIDIA VSS pattern used in `resnet.ipynb`.